# **Building a simple Transformer model using the concepts of Tokenization, Positional Embeddings (Sin/Cos waves), Transformers (Self-Attention) and Feed forward network (ReLU), by wrapping everything into a proper `PyTorch` Class Using `nn.Module`**

Why wrap it in a class?

Right now our code is scattered across multiple cells — embedding here, positional encoding there, attention somewhere else. In real models, everything is organized into modules — self-contained blocks you can reuse, stack, and train.

## Step 1 — What `nn.Module` and `nn.Linear` do

```python
# Quick refresher before we build

# nn.Module = a container for your model. It:
#   - Holds learnable parameters (weights)
#   - Knows how to do a forward pass (input → output)
#   - Lets PyTorch track gradients automatically

# nn.Linear(in_features, out_features) = a layer that does:
#   output = input @ weight.T + bias
#   It's basically matrix multiplication with LEARNABLE weights.
#   This replaces our manual W_Q = torch.randn(...) approach.
```
`nn.Linear` does `Q = final_embeddings @ W_Q`, — and it also manages the weights for us and makes them trainable automatically. We created `W_Q` manually with `torch.randn`. With `nn.Linear`, PyTorch creates it, tracks it, and updates it during training.

## Step 2 — The Embedding + Positional Encoding Module

In [1]:
import torch
import torch.nn as nn
import math

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=100):
        super().__init__()
        # The lookup table: vocab_size rows, d_model columns
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Positional encoding (not learned — fixed sin/cos)
        pe = torch.zeros(max_len, d_model)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                denominator = 10000 ** (i / d_model)
                pe[pos, i]     = math.sin(pos / denominator)
                pe[pos, i + 1] = math.cos(pos / denominator)

        # register_buffer = "save this tensor but DON'T train it"
        # because positional encodings are fixed, not learned
        self.register_buffer('pe', pe)

    def forward(self, token_ids):
        # Step 1: Look up token embeddings
        tok_emb = self.embedding(token_ids)

        # Step 2: Add positional encoding
        seq_len = token_ids.shape[0]
        output = tok_emb + self.pe[:seq_len]

        return output

## Step 3 — Self-Attention as a Module

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        # These replace our manual W_Q, W_K, W_V = torch.randn(...)
        # nn.Linear creates the weight matrix AND makes it trainable
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.d_k = d_k

    def forward(self, x):
        # x shape: (seq_len, d_model) — our final_embeddings

        # Step 1: Project into Q, K, V
        # Same as: Q = final_embeddings @ W_Q
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # Step 2: Dot product — "how relevant is each word to each other?"
        scores = Q @ K.T

        # Step 3: Scale — keep softmax in its useful zone
        scaled_scores = scores / math.sqrt(self.d_k)

        # Step 4: Softmax — convert to probabilities (each row sums to 1)
        attention_weights = torch.softmax(scaled_scores, dim=-1)

        # Step 5: Multiply by V — blend information based on attention
        output = attention_weights @ V

        return output, attention_weights

## Step 4 — Putting It All Together

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, vocab_size, d_model, d_k):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.self_attention = SelfAttention(d_model, d_k)

    def forward(self, token_ids):
        # Full pipeline: IDs → embeddings → positional → attention
        embeddings = self.token_embedding(token_ids)
        output, attention_weights = self.self_attention(embeddings)
        return output, attention_weights

## Step 5 — Run It

In [4]:
# Our sentence setup from before
text = "I love code and I love AI"
tokens = text.lower().split()
vocab = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocab)}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

print("Tokens:", tokens)
print("IDs:", token_ids)

# Create the model
model = TransformerBlock(
    vocab_size=len(vocab),   # 5 unique words
    d_model=4,               # embedding dimension
    d_k=3                    # attention dimension
)

# Run it!
output, attention_weights = model(token_ids)

print("\nOutput shape:", output.shape)          # (7, 3)
print("Attention matrix shape:", attention_weights.shape)  # (7, 7)

print("\nContextualized output:")
print(output)

print("\nAttention weights (who pays attention to whom):")
print(attention_weights)

Tokens: ['i', 'love', 'code', 'and', 'i', 'love', 'ai']
IDs: tensor([3, 4, 2, 1, 3, 4, 0])

Output shape: torch.Size([7, 3])
Attention matrix shape: torch.Size([7, 7])

Contextualized output:
tensor([[-0.1627, -0.0996,  0.5626],
        [-0.0979, -0.1107,  0.4928],
        [-0.1733, -0.0442,  0.4536],
        [-0.3155,  0.0057,  0.5632],
        [-0.2094, -0.1152,  0.6701],
        [-0.1939, -0.0868,  0.5859],
        [-0.2364, -0.0245,  0.5057]], grad_fn=<MmBackward0>)

Attention weights (who pays attention to whom):
tensor([[0.1495, 0.1248, 0.1308, 0.1540, 0.1540, 0.1386, 0.1482],
        [0.1122, 0.1173, 0.1449, 0.2233, 0.1459, 0.1155, 0.1409],
        [0.0862, 0.1734, 0.1983, 0.1947, 0.1071, 0.1175, 0.1228],
        [0.1240, 0.2289, 0.1774, 0.0886, 0.1066, 0.1593, 0.1152],
        [0.2221, 0.1082, 0.0891, 0.0902, 0.1852, 0.1605, 0.1447],
        [0.1544, 0.1371, 0.1300, 0.1348, 0.1577, 0.1458, 0.1401],
        [0.1054, 0.1935, 0.1895, 0.1410, 0.1091, 0.1370, 0.1246]],
       grad_f

## Step 6 — Inspect What the Model Learned

In [5]:
# Let's peek inside the model

print("=== Learnable parameters ===")
for name, param in model.named_parameters():
    print(f"{name}: shape {param.shape}, trainable: {param.requires_grad}")

print("\n=== Token Embedding Table ===")
print(model.token_embedding.embedding.weight)

print("\n=== Positional Encoding (NOT trainable) ===")
print(model.token_embedding.pe[:7])

=== Learnable parameters ===
token_embedding.embedding.weight: shape torch.Size([5, 4]), trainable: True
self_attention.W_Q.weight: shape torch.Size([3, 4]), trainable: True
self_attention.W_K.weight: shape torch.Size([3, 4]), trainable: True
self_attention.W_V.weight: shape torch.Size([3, 4]), trainable: True

=== Token Embedding Table ===
Parameter containing:
tensor([[ 0.1611, -0.3845,  0.3200, -0.3646],
        [-0.4777,  0.2280,  1.5162, -0.6135],
        [ 0.4944,  0.4596,  0.1930,  0.2364],
        [-0.6234, -0.4028, -1.0440, -0.6414],
        [ 0.5376, -0.8939, -1.1830,  0.4918]], requires_grad=True)

=== Positional Encoding (NOT trainable) ===
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992],
        [-0.9589,  0.2837,  0.0500,  0.9988],
        [-0.2794,  0.9602,  0.0600,  0.9982]])
